# PPO Self-Judge ORM Experiments
Compares PPO training with three reward modes on GSM8K:
1. **Deterministic** — binary 0/1 (baseline)
2. **Self-Judge** — log-likelihood from frozen reference model
3. **Combined** — weighted blend of both

Requires: T4 GPU, ~4GB VRAM (policy 0.5B + reference 0.5B + critic)

In [ ]:
!git clone https://github.com/madhu24raj/RLVR-Comparison.git 2>/dev/null || true
%cd RLVR-Comparison
!git pull
!pip install -q -r requirements.txt

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import sys, os
sys.path.insert(0, '.')
from ppo_specs.config import e2_7_config
from ppo_specs.run_e2_7 import run_e2_7

# T4-safe overrides: smaller batch + gradient checkpointing to fit
# policy + reference model in 15GB with activations
cfg = e2_7_config(seed=42)
cfg.reward_mode = "deterministic"
cfg.batch_size = 8
cfg.eval_batch_size = 4
cfg.gradient_checkpointing = True
cfg.experiment_name = "ppo_e2_7_seed42"

run_e2_7(cfg, compute_mc=False)

In [ ]:
cfg = e2_7_config(seed=42)
cfg.reward_mode = "self_judge"
cfg.batch_size = 8
cfg.eval_batch_size = 4
cfg.gradient_checkpointing = True
cfg.experiment_name = "ppo_e2_7_self_judge_seed42"

run_e2_7(cfg, compute_mc=False)

In [ ]:
cfg = e2_7_config(seed=42)
cfg.reward_mode = "combined"
cfg.self_judge_weight = 0.5
cfg.batch_size = 8
cfg.eval_batch_size = 4
cfg.gradient_checkpointing = True
cfg.experiment_name = "ppo_e2_7_combined_w0.5_seed42"

run_e2_7(cfg, compute_mc=False)

In [ ]:
import json
import matplotlib.pyplot as plt

configs = {
    "deterministic": "results/ppo_e2_7_seed42.json",
    "self_judge": "results/ppo_e2_7_self_judge_seed42.json",
    "combined": "results/ppo_e2_7_combined_w0.5_seed42.json",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for name, path in configs.items():
    try:
        with open(path) as f:
            data = json.load(f)
        steps = [d["step"] for d in data]
        test_acc = [d["test_accuracy"] for d in data]
        train_acc = [d["train_accuracy"] for d in data]
        reward_var = [d["reward_variance"] for d in data]

        axes[0].plot(steps, test_acc, label=name, marker="o")
        axes[1].plot(steps, train_acc, label=name, marker="o")
        axes[2].plot(steps, reward_var, label=name, marker="o")
    except FileNotFoundError:
        print(f"Skipping {name}: {path} not found")

axes[0].set_title("Held-out Accuracy")
axes[0].set_xlabel("Step")
axes[0].legend()

axes[1].set_title("Train Accuracy")
axes[1].set_xlabel("Step")
axes[1].legend()

axes[2].set_title("Reward Variance")
axes[2].set_xlabel("Step")
axes[2].legend()

plt.tight_layout()
plt.savefig("results/reward_mode_comparison.png", dpi=150)
plt.show()
print("Saved to results/reward_mode_comparison.png")

In [ ]:
if torch.cuda.is_available():
    print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    print(f"Current VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")